# Fix: `name` faltante en `orgattendees`

Algunos documentos de la colección `orgattendees` no tienen el campo `name` (aunque el schema lo marca como `required`). Este notebook lo reconstruye a partir de un campo dinámico de `registrationData`, que **varía por organización** porque el formulario de registro genera IDs de campo distintos por org.

Orgs a corregir en esta pasada:

| organizationId | campo de `registrationData` a usar como `name` |
|---|---|
| `69fb9e3f92bed35a91bceba6` | `field_1778512559238_0` |
| `6a14f181bb3953bbb32e391c` | `field_1779771568745_0` |

**Seguridad**: el notebook corre en modo **dry-run por defecto** (solo lee y muestra lo que haría). Para escribir en MongoDB hay que cambiar explícitamente `APPLY_CHANGES = True` en la celda de configuración y volver a correr las celdas de aplicación. Revisa el preview antes de aplicar.

## Cómo correrlo en Google Colab

1. Subí este archivo a Colab (`Archivo > Subir notebook`) o abrilo directo desde GitHub.
2. Para la connection string de Mongo, usá **Colab Secrets** (ícono de llave 🔑 en la barra lateral izquierda):
   - Agregá un secreto llamado `MONGO_URI` con la connection string completa.
   - Activá el toggle "Notebook access" para este notebook.
3. Corré las celdas en orden. Si no configuraste el secreto, la celda de conexión te va a pedir la URI de forma oculta (`getpass`) — no hace falta pegarla en texto plano en ninguna celda.

## 1. Setup

Instala `pymongo` (no viene preinstalado en Colab) y conecta a MongoDB.

In [ ]:
%pip install -q pymongo

In [ ]:
from getpass import getpass
from pymongo import MongoClient, UpdateOne
from bson import ObjectId

MONGO_URI = None

# Opción 1 (recomendada en Colab): leer desde Secrets (ícono de llave 🔑)
try:
    from google.colab import userdata
    MONGO_URI = userdata.get("MONGO_URI")
except Exception:
    pass

# Opción 2: si no hay secreto configurado (o no estás en Colab), pedirla de forma oculta
if not MONGO_URI:
    MONGO_URI = getpass("MONGO_URI (no se muestra en pantalla): ")

assert MONGO_URI, "Se necesita MONGO_URI para continuar."

client = MongoClient(MONGO_URI)
db = client.get_default_database()
attendees = db["orgattendees"]

print(f"Conectado a DB: {db.name}")
print(f"Total documentos en orgattendees: {attendees.count_documents({})}")

## 2. Configuración

`organizationId` en estos documentos se guarda como **string** (no ObjectId) — así viene en los ejemplos que compartiste. El filtro de abajo igual contempla ambas formas por si acaso.

In [ ]:
# organizationId -> campo de registrationData que contiene el nombre
ORG_FIELD_MAP = {
    "69fb9e3f92bed35a91bceba6": "field_1778512559238_0",
    "6a14f181bb3953bbb32e391c": "field_1779771568745_0",
}

# ⚠️ Mientras esté en False, el notebook SOLO muestra el preview (dry-run).
# Cambialo a True (y vuelve a correr las celdas de aplicación) para escribir en Mongo.
APPLY_CHANGES = False

## 3. Buscar documentos con `name` faltante

Considera "faltante" a: el campo no existe, es `null`, o es string vacío/solo-espacios.

In [ ]:
def org_id_filter(org_id: str):
    """organizationId puede venir como string o, en teoría, como ObjectId."""
    ids = [org_id]
    try:
        ids.append(ObjectId(org_id))
    except Exception:
        pass
    return {"organizationId": {"$in": ids}}


MISSING_NAME_FILTER = {
    "$or": [
        {"name": {"$exists": False}},
        {"name": None},
        {"name": ""},
        {"name": {"$regex": r"^\s*$"}},
    ]
}


def find_missing_for_org(org_id: str):
    query = {"$and": [org_id_filter(org_id), MISSING_NAME_FILTER]}
    return list(attendees.find(query))


candidates_by_org = {org_id: find_missing_for_org(org_id) for org_id in ORG_FIELD_MAP}

for org_id, docs in candidates_by_org.items():
    print(f"Org {org_id}: {len(docs)} documento(s) sin name")

## 4. Preview (dry-run)

Calcula el `name` propuesto para cada documento a partir del campo mapeado, hace `.strip()` del valor (los datos de ejemplo traen espacios al final, ej. `"PATRICIA ISABEL "`), y separa los que **no** se pueden resolver (el campo no está en `registrationData` o viene vacío) para revisarlos a mano — esos se omiten de la aplicación.

In [ ]:
def build_plan_for_org(org_id: str, field_key: str, docs: list):
    resolved = []  # (doc, new_name)
    unresolved = []  # doc sin valor usable en el campo mapeado

    for doc in docs:
        raw = (doc.get("registrationData") or {}).get(field_key)
        new_name = raw.strip() if isinstance(raw, str) else None
        if new_name:
            resolved.append((doc, new_name))
        else:
            unresolved.append(doc)

    return resolved, unresolved


plans = {}  # org_id -> (resolved, unresolved)

for org_id, field_key in ORG_FIELD_MAP.items():
    docs = candidates_by_org[org_id]
    resolved, unresolved = build_plan_for_org(org_id, field_key, docs)
    plans[org_id] = (resolved, unresolved)

    print(f"\n=== Org {org_id} (campo: {field_key}) ===")
    print(f"  A corregir: {len(resolved)}   |  Sin valor usable (omitidos): {len(unresolved)}")

    print("  Preview (hasta 10):")
    for doc, new_name in resolved[:10]:
        print(f"    {doc.get('email', doc['_id'])!s:40s} -> name = {new_name!r}")

    if unresolved:
        print("  ⚠️ Omitidos — revisar a mano:")
        for doc in unresolved[:10]:
            print(f"    _id={doc['_id']} email={doc.get('email')} registrationData.{field_key}={doc.get('registrationData', {}).get(field_key)!r}")

## 5. Aplicar cambios

Solo escribe si `APPLY_CHANGES = True` (celda de configuración, arriba). Usa `bulk_write` por org con `ordered=False` para que un error puntual no frene al resto.

In [ ]:
if not APPLY_CHANGES:
    print("🔍 DRY-RUN — no se escribió nada. Cambia APPLY_CHANGES = True arriba para aplicar.")
else:
    print("✏️  APPLY — escribiendo cambios en MongoDB...\n")
    total_applied = 0

    for org_id, (resolved, _unresolved) in plans.items():
        if not resolved:
            continue

        ops = [
            UpdateOne({"_id": doc["_id"]}, {"$set": {"name": new_name}})
            for doc, new_name in resolved
        ]
        result = attendees.bulk_write(ops, ordered=False)
        modified = result.modified_count
        total_applied += modified
        print(f"Org {org_id}: {modified} documento(s) actualizados")

    print(f"\n✅ Total actualizado: {total_applied}")

## 6. Verificación

Vuelve a correr la búsqueda de "sin name" para confirmar que ya no quedan pendientes (salvo los que quedaron en `unresolved` por no tener valor usable en el campo).

In [ ]:
for org_id in ORG_FIELD_MAP:
    remaining = find_missing_for_org(org_id)
    print(f"Org {org_id}: {len(remaining)} documento(s) todavía sin name")